<a href="https://colab.research.google.com/github/lakshyareddycse/PTOJECTS/blob/main/End_to_End_E_Commerce_Data_Pipeline_%26_Dimensional_Warehouse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Environment Setup
!pip install pyspark -q
!apt-get update -q
!apt-get install postgresql postgresql-contrib -q
!service postgresql start

# Create Database and User
!sudo -u postgres psql -c "CREATE USER colab WITH PASSWORD 'colab';"
!sudo -u postgres psql -c "CREATE DATABASE ecommerce_db OWNER colab;"
!sudo -u postgres psql -c "GRANT ALL PRIVILEGES ON DATABASE ecommerce_db TO colab;"

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,040 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:9 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,320 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.3 MB]
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [52.1 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/restricted amd64 Packages [2,135

In [ ]:
# Cell 2: Ingestion & Spark Initialization (Direct Key Method)
import os
import zipfile
from pyspark.sql import SparkSession

# Set your Kaggle credentials directly
os.environ['KAGGLE_USERNAME'] = "glakshyareddy"
os.environ['KAGGLE_KEY'] = "KGAT_cafd91b1b3c4ec710f68c47def3606a5"

# 1. Install Kaggle CLI and download dataset
!pip install kaggle -q
!kaggle datasets download -d mashlyn/online-retail-ii-uci -p /content/

# 2. Unzip dataset
with zipfile.ZipFile("/content/online-retail-ii-uci.zip", "r") as zip_ref:
    zip_ref.extractall("/content/")

# 3. Download PostgreSQL JDBC Driver for Spark
!wget -q https://jdbc.postgresql.org/download/postgresql-42.6.0.jar -O postgresql-42.6.0.jar

# 4. Initialize PySpark Session with PostgreSQL Driver
spark = SparkSession.builder \
    .appName("ECommerceDataPipeline") \
    .config("spark.driver.extraClassPath", "postgresql-42.6.0.jar") \
    .getOrCreate()

# 5. Load CSV into Spark DataFrame
df_raw = spark.read.csv("/content/online_retail_II.csv", header=True, inferSchema=True)

print("Raw Data Count:", df_raw.count())
df_raw.show(5)

Dataset URL: https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci
License(s): CC0-1.0
100% 14.5M/14.5M [00:00<00:00, 70.0MB/s]

Raw Data Count: 1067371
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|         Description|Quantity|        InvoiceDate|Price|Customer ID|       Country|
+-------+---------+--------------------+--------+-------------------+-----+-----------+--------------+
| 489434|    85048|15CM CHRISTMAS GL...|      12|2009-12-01 07:45:00| 6.95|    13085.0|United Kingdom|
| 489434|   79323P|  PINK CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|   79323W| WHITE CHERRY LIGHTS|      12|2009-12-01 07:45:00| 6.75|    13085.0|United Kingdom|
| 489434|    22041|"RECORD FRAME 7""...|      48|2009-12-01 07:45:00|  2.1|    13085.0|United Kingdom|
| 489434|    21232|STRAWBERRY CERAMI...|      24|2009-12-01 07:45:00| 1.25|    13085.0|United Kingdom|
+-------+-------

In [ ]:
# Cell 3: Data Cleaning, Transformations & Deduplication
from pyspark.sql.functions import col, to_timestamp, year, month, dayofmonth, hour, round

# 1. Data Cleaning & Normalization
df_clean = df_raw.filter(
    (col("Quantity") > 0) &
    (col("Price") > 0) &
    (col("Customer ID").isNotNull()) &
    (col("Invoice").isNotNull())
).withColumn("InvoiceDate", to_timestamp(col("InvoiceDate"), "yyyy-MM-dd HH:mm:ss")) \
 .withColumn("CustomerID", col("Customer ID").cast("integer"))

# 2. Dimension Table: Customer (Deduplicated on customer_id)
dim_customer = df_clean.select("CustomerID", "Country") \
    .dropDuplicates(["CustomerID"]) \
    .withColumnRenamed("CustomerID", "customer_id") \
    .withColumnRenamed("Country", "country")

# 3. Dimension Table: Product (Deduplicated on product_id)
dim_product = df_clean.select("StockCode", "Description", "Price") \
    .dropDuplicates(["StockCode"]) \
    .withColumnRenamed("StockCode", "product_id") \
    .withColumnRenamed("Description", "product_name") \
    .withColumnRenamed("Price", "unit_price")

# 4. Dimension Table: Time
dim_time = df_clean.select("InvoiceDate").distinct() \
    .withColumnRenamed("InvoiceDate", "time_id") \
    .withColumn("year", year("time_id")) \
    .withColumn("month", month("time_id")) \
    .withColumn("day", dayofmonth("time_id")) \
    .withColumn("hour", hour("time_id"))

# 5. Fact Table: Sales
fact_sales = df_clean.withColumn("total_amount", round(col("Quantity") * col("Price"), 2)) \
    .select(
        col("Invoice").alias("invoice_no"),
        col("CustomerID").alias("customer_id"),
        col("StockCode").alias("product_id"),
        col("InvoiceDate").alias("time_id"),
        col("Quantity").alias("quantity"),
        col("Price").alias("unit_price"),
        col("total_amount")
    )

print("Deduplicated Dimension and Fact tables created successfully.")

Deduplicated Dimension and Fact tables created successfully.


In [ ]:
# Cell 4: Database Schema DDL & JDBC Data Load
import psycopg2

# Connect to PostgreSQL to execute DDL statements
conn = psycopg2.connect(
    dbname="ecommerce_db", user="colab", password="colab", host="localhost", port="5432"
)
cursor = conn.cursor()

# Drop existing tables if re-running
cursor.execute("""
DROP TABLE IF EXISTS fact_sales CASCADE;
DROP TABLE IF EXISTS dim_customer CASCADE;
DROP TABLE IF EXISTS dim_product CASCADE;
DROP TABLE IF EXISTS dim_time CASCADE;
""")

# DDL Statements for Star Schema
cursor.execute("""
CREATE TABLE dim_customer (
    customer_id INT PRIMARY KEY,
    country VARCHAR(100)
);

CREATE TABLE dim_product (
    product_id VARCHAR(50) PRIMARY KEY,
    product_name VARCHAR(255),
    unit_price NUMERIC(10, 2)
);

CREATE TABLE dim_time (
    time_id TIMESTAMP PRIMARY KEY,
    year INT,
    month INT,
    day INT,
    hour INT
);

CREATE TABLE fact_sales (
    sales_id SERIAL PRIMARY KEY,
    invoice_no VARCHAR(50),
    customer_id INT,
    product_id VARCHAR(50),
    time_id TIMESTAMP,
    quantity INT,
    unit_price NUMERIC(10, 2),
    total_amount NUMERIC(10, 2),
    FOREIGN KEY (customer_id) REFERENCES dim_customer(customer_id),
    FOREIGN KEY (time_id) REFERENCES dim_time(time_id)
);
""")
conn.commit()
cursor.close()
conn.close()

# PySpark JDBC Configuration
db_url = "jdbc:postgresql://localhost:5432/ecommerce_db"
properties = {
    "user": "colab",
    "password": "colab",
    "driver": "org.postgresql.Driver"
}

# Load PySpark DataFrames into PostgreSQL
dim_customer.write.jdbc(url=db_url, table="dim_customer", mode="append", properties=properties)
dim_product.write.jdbc(url=db_url, table="dim_product", mode="append", properties=properties)
dim_time.write.jdbc(url=db_url, table="dim_time", mode="append", properties=properties)
fact_sales.write.jdbc(url=db_url, table="fact_sales", mode="append", properties=properties)

print("Data successfully loaded into PostgreSQL Data Warehouse.")

Data successfully loaded into PostgreSQL Data Warehouse.


In [ ]:
# Cell 5: Data Warehouse Verification & SQL Business Analytics
import psycopg2
import pandas as pd

# 1. Establish Database Connection
conn = psycopg2.connect(
    dbname="ecommerce_db",
    user="colab",
    password="colab",
    host="localhost",
    port="5432"
)

# 2. Verify Row Counts in Target Star Schema Tables
table_counts_query = """
SELECT 'dim_customer' AS table_name, COUNT(*) AS total_rows FROM dim_customer
UNION ALL
SELECT 'dim_product' AS table_name, COUNT(*) AS total_rows FROM dim_product
UNION ALL
SELECT 'dim_time' AS table_name, COUNT(*) AS total_rows FROM dim_time
UNION ALL
SELECT 'fact_sales' AS table_name, COUNT(*) AS total_rows FROM fact_sales;
"""

# 3. Analytical Query 1: Top 5 Revenue-Generating Countries
top_countries_query = """
SELECT
    c.country,
    ROUND(SUM(f.total_amount), 2) AS total_revenue,
    COUNT(DISTINCT f.invoice_no) AS total_orders
FROM fact_sales f
JOIN dim_customer c ON f.customer_id = c.customer_id
GROUP BY c.country
ORDER BY total_revenue DESC
LIMIT 5;
"""

# 4. Analytical Query 2: Monthly Sales Revenue Trend
monthly_revenue_query = """
SELECT
    t.year,
    t.month,
    ROUND(SUM(f.total_amount), 2) AS monthly_revenue,
    COUNT(DISTINCT f.invoice_no) AS total_orders
FROM fact_sales f
JOIN dim_time t ON f.time_id = t.time_id
GROUP BY t.year, t.month
ORDER BY t.year, t.month;
"""

print("================ TABLE ROW COUNTS ================")
print(pd.read_sql(table_counts_query, conn))

print("\n============ TOP 5 COUNTRIES BY REVENUE ============")
print(pd.read_sql(top_countries_query, conn))

print("\n============ MONTHLY REVENUE TRENDS ============")
print(pd.read_sql(monthly_revenue_query, conn))

# Close connection
conn.close()

================ TABLE ROW COUNTS ================


/tmp/ipykernel_776/3940493812.py:52: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql(table_counts_query, conn))
/tmp/ipykernel_776/3940493812.py:55: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql(top_countries_query, conn))


     table_name  total_rows
0  dim_customer        5878
1   dim_product        4631
2      dim_time       34585
3    fact_sales      805549

============ TOP 5 COUNTRIES BY REVENUE ============
          country  total_revenue  total_orders
0  United Kingdom    14722423.65         33539
1            EIRE      622354.96           569
2     Netherlands      554232.34           228
3         Germany      432182.44           790
4          France      353579.39           610

============ MONTHLY REVENUE TRENDS ============


/tmp/ipykernel_776/3940493812.py:58: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql(monthly_revenue_query, conn))


    year  month  monthly_revenue  total_orders
0   2009     12        686654.16          1512
1   2010      1        557319.06          1011
2   2010      2        506371.06          1104
3   2010      3        699608.99          1524
4   2010      4        594609.19          1329
5   2010      5        599985.79          1377
6   2010      6        639066.58          1497
7   2010      7        591636.74          1381
8   2010      8        604242.65          1293
9   2010      9        831615.00          1689
10  2010     10       1036680.00          2133
11  2010     11       1172336.04          2587
12  2010     12        884591.89          1400
13  2011      1        569445.04           987
14  2011      2        447137.35           997
15  2011      3        595500.76          1321
16  2011      4        469200.36          1149
17  2011      5        678594.56          1555
18  2011      6        661213.69          1393
19  2011      7        600091.01          1331
20  2011     